# Full Reverify — Google Colab

Verify lại **toàn bộ** nhãn trong `hasaki_labelled_clean.json` (~2011 mẫu) bằng GPT-4o verifier + guardrails.

**Không gọi lại annotator** — giữ nhãn hiện có, chỉ chạy verifier (`force=True`).

**Output:** `data/raw/hasaki/hasaki_labelled_full_verified.json`

**Sau khi xong:**
```bash
python scripts/build_d2_verified_dataset.py --mode full
python scripts/sample_d3_human_gold.py --n 300 --seed 42
python scripts/build_d1_d2_train.py
```

Chạy **tuần tự từ §1 → §6**. Checkpoint mỗi 50 mẫu — `RESUME=True` nếu disconnect.


## 1. Cài đặt packages

In [ ]:
!pip -q install pymongo certifi openai sentence-transformers scikit-learn numpy tqdm


## 2. Mount Google Drive + INTENT_REPO

Repo mặc định: `/content/drive/MyDrive/intent_kb`


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DEFAULT_DRIVE_REPO = "/content/drive/MyDrive/intent_kb"

if IN_COLAB:
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")
    os.environ.setdefault("INTENT_REPO", DEFAULT_DRIVE_REPO)
else:
    os.environ.setdefault("INTENT_REPO", str(Path.cwd().resolve()))

REPO_ROOT = Path(os.environ["INTENT_REPO"]).resolve()
print("INTENT_REPO:", REPO_ROOT)
print("clean exists:", (REPO_ROOT / "data/raw/hasaki/hasaki_labelled_clean.json").exists())
print("pipeline exists:", (REPO_ROOT / "data/code/labeling_pipeline.py").exists())


## 3. API keys

In [ ]:
import getpass

if not os.environ.get("MONGODB_URI", "").strip():
    os.environ["MONGODB_URI"] = getpass.getpass("MONGODB_URI: ").strip()
if not os.environ.get("OPENAI_API_KEY", "").strip():
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ").strip()

os.environ.setdefault("ENABLE_VERIFIER", "1")
os.environ.setdefault("VERIFIER_MODEL", "gpt-4o")

print("MONGODB_URI set:", bool(os.environ.get("MONGODB_URI")))
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))


## 4. Load `labeling_pipeline.py`

Import pipeline thật từ repo — **phải chạy cell này trước §6**.


In [ ]:
sys.path.insert(0, str(REPO_ROOT / "data" / "code"))

# Reload nếu đã import lần trước (dev/debug)
if "labeling_pipeline" in sys.modules:
    del sys.modules["labeling_pipeline"]

import labeling_pipeline as lp
from labeling_pipeline import (
    ENABLE_VERIFIER,
    VERIFIER_MODEL,
    apply_verifier,
    db,
    enrich_sample,
    save_annotation_with_guardrails,
    union_retrieval,
)

print("MongoDB:", db.name)
print("Verifier:", ENABLE_VERIFIER, "| model:", VERIFIER_MODEL)


## 5. Cấu hình chạy

In [ ]:
import json

INPUT_FILE = REPO_ROOT / "data/raw/hasaki/hasaki_labelled_clean.json"
OUTPUT_FILE = REPO_ROOT / "data/raw/hasaki/hasaki_labelled_full_verified.json"
CHECKPOINT_FILE = REPO_ROOT / "data/raw/hasaki/hasaki_labelled_full_verified.checkpoint.json"

LIMIT = 0          # 0 = full ~2011; 10 = smoke test
RESUME = False     # True = tiếp tục từ checkpoint
CHECKPOINT_EVERY = 50

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Không tìm thấy {INPUT_FILE}")

print("Input :", INPUT_FILE)
print("Output:", OUTPUT_FILE)
print("Checkpoint:", CHECKPOINT_FILE)
print(f"LIMIT={LIMIT}, RESUME={RESUME}, CHECKPOINT_EVERY={CHECKPOINT_EVERY}")


## 6. Chạy full reverify

Giữ nhãn từ `hasaki_labelled_clean.json` → retrieval → verifier → guardrails (`persist=False`).


In [ ]:
from tqdm.notebook import tqdm

_SAMPLE_FIELDS = ("sample_id", "sentence", "category", "product_id", "product_name", "brand")


def intent_from_ann(ann_intent, fallback_intent=None):
    fb = fallback_intent or {}
    l3_raw = (ann_intent or {}).get("level_3")
    if isinstance(l3_raw, list):
        l3 = l3_raw[0] if l3_raw else ""
    elif isinstance(l3_raw, str):
        l3 = l3_raw
    else:
        fb3 = fb.get("level_3")
        l3 = fb3[0] if isinstance(fb3, list) and fb3 else (fb3 or "")
    return {
        "level_1": (ann_intent or {}).get("level_1") or fb.get("level_1", ""),
        "level_2": (ann_intent or {}).get("level_2") or fb.get("level_2", ""),
        "level_3": l3 or "",
    }


def pred_from_labelled_item(item):
    intent = item.get("intent_3_level") or {}
    l3_val = intent.get("level_3")
    if isinstance(l3_val, list):
        l3_val = l3_val[0] if l3_val else ""
    return {
        "L1": intent.get("level_1", ""),
        "L2": intent.get("level_2", ""),
        "L3": l3_val or "",
        "confidence": float(item.get("confidence") or 0),
        "reasoning": item.get("reasoning", ""),
        "is_new_label": False,
    }


def save_checkpoint(path, results, stats, meta):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump({**meta, "stats": stats, "num_done": len(results), "results": results},
                  f, ensure_ascii=False, indent=2)
    tmp.replace(path)


def load_checkpoint(path):
    if not path.exists():
        return [], {}, set()
    try:
        with path.open(encoding="utf-8") as f:
            doc = json.load(f)
        results = doc.get("results", [])
        stats = doc.get("stats", {})
        done_ids = {r["sample_id"] for r in results if r.get("sample_id")}
        print(f"Resumed checkpoint: {len(results)} done")
        return results, stats, done_ids
    except (json.JSONDecodeError, KeyError) as e:
        print(f"WARNING: corrupt checkpoint ({e}) — start fresh")
        return [], {}, set()


with INPUT_FILE.open(encoding="utf-8") as f:
    _in = json.load(f)
all_items = _in.get("results", _in) if isinstance(_in, dict) else _in
all_items = [x for x in all_items if x.get("sample_id")]

results, stats, done_ids = [], {}, set()
if RESUME:
    results, stats, done_ids = load_checkpoint(CHECKPOINT_FILE)

todo = [x for x in all_items if x["sample_id"] not in done_ids]
if LIMIT > 0:
    todo = todo[:LIMIT]

meta = {
    "source_file": str(INPUT_FILE),
    "reverify_scope": "full_all_samples",
    "reverify_config": {
        "enable_verifier": ENABLE_VERIFIER,
        "verifier_model": VERIFIER_MODEL,
        "force_verify": True,
        "notebook": "notebooks/run_full_reverify_colab.ipynb",
    },
}

print(f"Total: {len(all_items)} | checkpoint skip: {len(done_ids)} | todo: {len(todo)}")

for idx, item in enumerate(tqdm(todo, desc="reverify")):
    sample = enrich_sample({k: item.get(k, "") for k in _SAMPLE_FIELDS})
    intent_before = item.get("intent_3_level") or {}
    pred = pred_from_labelled_item(item)
    qa_status_before = item.get("qa_status")
    verify_meta = None
    ann = {}
    status = qa_status_before
    reverify_error = None

    try:
        cands = union_retrieval(
            db,
            sample.get("sentence", ""),
            category=sample.get("category"),
            sample=sample,
        )
        verify_meta = apply_verifier(sample, pred, cands, force=True)
        ann = save_annotation_with_guardrails(
            db, sample, pred, verify_meta=verify_meta, persist=False
        )
        status = ann.get("qa_status", status)
    except Exception as e:
        reverify_error = str(e)
        status = "reverify_error"

    stats[status] = stats.get(status, 0) + 1
    if verify_meta is not None:
        vkey = f"verify_{str(verify_meta.get('decision', '')).lower()}"
        stats[vkey] = stats.get(vkey, 0) + 1

    intent_after = intent_from_ann(ann.get("intent"), fallback_intent=intent_before)
    results.append({
        **item,
        "qa_status_before_reverify": qa_status_before,
        "intent_3_level_before_reverify": intent_before,
        "intent_3_level": intent_after,
        "confidence": ann.get("confidence", pred.get("confidence")),
        "qa_status": status,
        "reasoning": pred.get("reasoning", item.get("reasoning")),
        "verifier_decision": (verify_meta or {}).get("decision"),
        "verifier_reason": (verify_meta or {}).get("reason"),
        "confidence_before_verify": (verify_meta or {}).get("confidence_before_verify"),
        "reverify_error": reverify_error,
    })

    if (idx + 1) % CHECKPOINT_EVERY == 0 or idx == len(todo) - 1:
        save_checkpoint(CHECKPOINT_FILE, results, stats, meta)

out_payload = {
    **meta,
    "num_samples_total": len(all_items),
    "num_samples": len(results),
    "stats": stats,
    "results": results,
}
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    json.dump(out_payload, f, ensure_ascii=False, indent=2)

print("\n=== Stats ===")
for k, v in sorted(stats.items()):
    print(f"  {k}: {v}")
print(f"\nSaved: {OUTPUT_FILE}")


## 7. Bước tiếp theo

```bash
python scripts/build_d2_verified_dataset.py --mode full
python scripts/sample_d3_human_gold.py --n 300 --seed 42
python scripts/build_d1_d2_train.py
```

Rồi mở `notebooks/evaluation_d1_d2_d3.ipynb`.
